In [4]:
# ===============================
# 1. Setup Imports & Path
# ===============================

import sys
import os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.model_selection import cross_val_score

from src.data_preprocessing import load_data, clean_data, prepare_train_test_data

# ===============================
# 2. Load & Prepare Data
# ===============================

df = load_data("../data/raw/telco_churn.csv")
df = clean_data(df)

X_train, X_test, y_train, y_test, preprocessor = prepare_train_test_data(df)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ===============================
# 3. Train Models
# ===============================

log_model = LogisticRegression(max_iter=1000, class_weight="balanced")
tree_model = DecisionTreeClassifier(random_state=42)
rf_model = RandomForestClassifier(random_state=42)

log_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

# ===============================
# 4. Evaluation Function
# ===============================

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

# ===============================
# 5. Evaluate All Models
# ===============================

models = {
    "Logistic Regression": log_model,
    "Decision Tree": tree_model,
    "Random Forest": rf_model
}

for name, model in models.items():
    metrics = evaluate(model, X_test, y_test)
    print(f"\n{name}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

# ===============================
# 6. Cross-Validation (ROC-AUC)
# ===============================

def cross_validate(model, X_train, y_train):
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
    return scores.mean()

print("\nCross-Validation ROC-AUC")
print("Logistic:", cross_validate(log_model, X_train, y_train))
print("Random Forest:", cross_validate(rf_model, X_train, y_train))

# ===============================
# 7. Threshold Tuning (Random Forest)
# ===============================

rf_probs = rf_model.predict_proba(X_test)[:, 1]

thresholds = [0.5, 0.4, 0.35, 0.3, 0.25]

print("\nThreshold Tuning (Random Forest)")
for t in thresholds:
    preds = (rf_probs >= t).astype(int)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    print(f"\nThreshold: {t}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

Train shape: (5634, 8465)
Test shape: (1409, 8465)

Logistic Regression
Accuracy: 0.7566
Precision: 0.5308
Recall: 0.7139
F1 Score: 0.6089
ROC-AUC: 0.8333

Decision Tree
Accuracy: 0.7637
Precision: 0.5574
Recall: 0.5321
F1 Score: 0.5445
ROC-AUC: 0.6897

Random Forest
Accuracy: 0.7963
Precision: 0.6654
Recall: 0.4679
F1 Score: 0.5495
ROC-AUC: 0.8396

Cross-Validation ROC-AUC
Logistic: 0.8398114927484069
Random Forest: 0.847349527779453

Threshold Tuning (Random Forest)

Threshold: 0.5
Precision: 0.6522
Recall: 0.4813
F1 Score: 0.5538

Threshold: 0.4
Precision: 0.6026
Recall: 0.6123
F1 Score: 0.6074

Threshold: 0.35
Precision: 0.5805
Recall: 0.6845
F1 Score: 0.6282

Threshold: 0.3
Precision: 0.5416
Recall: 0.7139
F1 Score: 0.6159

Threshold: 0.25
Precision: 0.5315
Recall: 0.7888
F1 Score: 0.6351


In [1]:
# ======================================================
# 1️⃣ Fix Project Path (so src can be imported)
# ======================================================

import sys
import os

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project Root:", project_root)

# ======================================================
# 2️⃣ Import Preprocessing Functions
# ======================================================

from src.data_preprocessing import load_data, clean_data, prepare_train_test_data

# ======================================================
# 3️⃣ Load Raw Dataset
# ======================================================

data_path = "../data/raw/telco_churn.csv"

df = load_data(data_path)

print("\nRaw Shape:", df.shape)

# ======================================================
# 4️⃣ Clean Dataset
# ======================================================

df = clean_data(df)

print("\nColumns After Cleaning:")
print(df.columns.tolist())

print("\nCleaned Shape:", df.shape)

# ======================================================
# 5️⃣ Prepare Train-Test Data
# ======================================================

X_train, X_test, y_train, y_test, preprocessor = prepare_train_test_data(df)

print("\nTrain Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

# ======================================================
# 6️⃣ Sanity Check – Feature Count
# ======================================================

print("\nTotal Features After Encoding:", X_train.shape[1])

Project Root: C:\Users\91910\Month-4 Machine Learning Fundamentals

Raw Shape: (7043, 33)

Columns After Cleaning:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'CLTV']

Cleaned Shape: (7043, 22)

Train Shape: (5634, 46)
Test Shape: (1409, 46)

Total Features After Encoding: 46
